# 01 · Data preprocessing
Cleans the raw 30-Jun-2026 customer snapshot (`input 1`) and writes the cleaned dataset (**artifact 2**) and a data-quality log (**artifact 3**).

**Run order:** 01 → 02 → 03 → 04

In [ ]:
import numpy as np
import pandas as pd

import pipeline_config as cfg

pd.set_option("display.max_columns", 40)
dq_log = []


def log(step, rows_affected, action):
    dq_log.append({"step": step, "rows_affected": int(rows_affected), "action": action})
    print(f"{step:<28} {rows_affected:>6,} rows  -> {action}")

## 1. Load and profile

In [ ]:
raw = pd.read_csv(cfg.RAW_FILE)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns | unique customers: {raw['customer_id'].nunique():,}")
print(f"Target rate: {raw[cfg.TARGET].mean():.1%}")
raw.head()

In [ ]:
missing = raw.isna().mean().mul(100).round(1)
missing[missing > 0].sort_values(ascending=False).to_frame("% missing")

In [ ]:
raw.describe().T[["min", "50%", "max"]]

## 2. Duplicates (CRM merge)

In [ ]:
df = raw.copy()
n_exact = df.duplicated().sum()
df = df.drop_duplicates()
log("exact duplicates", n_exact, "dropped")

n_conflict = df["customer_id"].duplicated().sum()
df = df.drop_duplicates(subset="customer_id", keep="first")
log("conflicting customer rows", n_conflict, "kept first record (no update timestamp)")

assert df["customer_id"].is_unique
print(f"{len(df):,} customers")

## 3. Dates and tenure

In [ ]:
df["customer_since"] = pd.to_datetime(df["customer_since"], format="%Y-%m-%d")

In [ ]:
snapshot = pd.Timestamp(cfg.SNAPSHOT_DATE)
months = (snapshot.year - df["customer_since"].dt.year) * 12 + (snapshot.month - df["customer_since"].dt.month)
bad_tenure = df["tenure_months"] < 0
df.loc[bad_tenure, "tenure_months"] = months[bad_tenure]
log("negative tenure", bad_tenure.sum(), "recomputed from customer_since")

## 4. Category labels

In [ ]:
REGION_FIXES = {"n.": "north", "w.": "west"}
PRODUCT_FIXES = {"current acct": "current account", "savings": "savings account",
                 "cc": "credit card", "p. loan": "personal loan"}

orig_region, orig_product = df["region"].copy(), df["primary_product"].copy()
df["region"] = df["region"].str.lower().replace(REGION_FIXES).str.title()
df["primary_product"] = df["primary_product"].str.strip().str.lower().replace(PRODUCT_FIXES).str.title()

assert df["region"].nunique() == 5, f"unexpected regions: {sorted(df['region'].unique())}"
assert df["primary_product"].nunique() == 5, f"unexpected products: {sorted(df['primary_product'].unique())}"
log("region labels", (orig_region != df["region"]).sum(), "normalised to 5 regions")
log("product labels", (orig_product != df["primary_product"]).sum(), "normalised to 5 products")

## 5. Impossible and sentinel values

In [ ]:
bad_age = ~df["age"].between(18, 100)
df.loc[bad_age, "age"] = np.nan
log("age outside 18-100", bad_age.sum(), "set to missing (system defaults 0 / 999)")

bad_balance = df["avg_monthly_balance"] == -99999
df.loc[bad_balance, "avg_monthly_balance"] = np.nan
log("balance = -99999", bad_balance.sum(), "set to missing (sentinel)")

## 6. Missing values

In [ ]:
df.groupby("customer_segment")["monthly_income"].apply(lambda s: s.isna().mean()).map("{:.1%}".format)

In [ ]:
# not missing at random (Mass segment) -> keep the signal as a flag, impute within segment
df["income_missing"] = df["monthly_income"].isna().astype(int)
df["monthly_income"] = df["monthly_income"].fillna(
    df.groupby("customer_segment")["monthly_income"].transform("median"))
log("monthly_income missing", df["income_missing"].sum(), "flag + segment median")

# structural: open rate is only tracked when emails were sent (opted-in customers)
print(pd.crosstab(df["marketing_opt_in"], df["email_open_rate"].isna().rename("open_rate_missing")))
n = df["email_open_rate"].isna().sum()
df["email_open_rate"] = df["email_open_rate"].fillna(0.0)
log("email_open_rate missing", n, "0 = not emailed (opted out)")

df["never_contacted"] = df["days_since_last_contact"].isna().astype(int)
df["days_since_last_contact"] = df["days_since_last_contact"].fillna(365)
log("days_since_last_contact", df["never_contacted"].sum(), "never contacted -> flag + 365")

## 7. Save

In [ ]:
df.to_csv(cfg.CLEAN_FILE, index=False)
pd.DataFrame(dq_log).to_csv(cfg.DQ_LOG_FILE, index=False)
print(f"Saved {len(df):,} rows -> {cfg.CLEAN_FILE.name}")
print(f"Saved {len(dq_log)} log entries -> {cfg.DQ_LOG_FILE.name}")
print("Still missing (handled in the model pipelines):", df.isna().sum()[lambda s: s > 0].to_dict())